# Measuring fan blade width with a laser and a photodiode

This notebook puts the ideas from `nidaqmx_tutorial.ipynb` into a small experiment.
A laser beam passes through a spinning fan onto a photodiode. Each time a blade passes,
it blocks the beam and the photodiode signal drops. We record a short snippet of signal
around every blade pass, overlay 1000 of them, and fit Gaussians to the edges to measure:

* the **pulse width** τ, i.e. how long a blade takes to pass the beam
* the **blade period** T, i.e. the time between one blade and the next

From these, together with the number of blades and where the beam crosses the fan, we get the blade width.

```
          laser ───────────────► ┃ ◄─ fan blade ─► photodiode ──► AI0
                                  (beam at radius r from the hub)

 photodiode   ─────┐     ┌─────────┐     ┌─────────┐     ┌────
 signal            │     │         │     │         │     │
                   └─────┘         └─────┘         └─────┘
                   ◄─ τ ─►
                   ◄────── T ──────►
```

The fan and the laser run from their **own power supplies**, at a fixed speed and
brightness. The DAQ only records: the photodiode is the only thing connected to it.

### The physics

The fan has `N` blades, so it turns once every `N·T` seconds. Its rotation frequency is

$$f = \frac{1}{N\,T}$$

At radius $r$ the blade moves at $v = 2\pi r f$, so the blade width is

$$w = v\,\tau = \frac{2\pi r\,\tau}{N\,T}$$

### Why Gaussians?

A laser beam's brightness falls off as a Gaussian from its centre. As a straight blade edge
sweeps across the beam, the light it blocks rises smoothly (as a cumulative Gaussian), and
the **slope** of the photodiode signal is the Gaussian itself, peaking at the moment the edge
crosses the centre of the beam. Fitting a Gaussian to the slope at each edge gives:

* its **centre**: when the edge crossed the beam's centre, to a fraction of a sample. The time
  between the arriving and leaving centres is τ. Both are measured at the beam's centre, so
  the beam's size cancels out of $w$.
* its **width** σ (in time): the beam's 1/e² diameter is $d = 4\,v\,\sigma$. This is the
  *knife-edge* method used to measure laser spots on real microscopes.

$w$ depends only on the **ratio** τ/T. We don't need to know the fan's speed in advance,
because T is measured in the same recording, so small changes in fan speed between
recordings shouldn't change the result. Section 5 checks this, and also checks that the
sample rate is high enough.

### Which tutorial ideas are used where

| Step | Tutorial idea |
|---|---|
| Live view for aligning the beam | Continuous AI loop / live plot (4a, 4b) |
| Recording the blade passes | Continuous AI read in chunks (4a) |
| Checking the sample rate | Choosing `RATE` for hardware-timed AI (1b) |
| Analysis | numpy, plus `scipy.optimize.curve_fit` for the Gaussians |

### Wiring and safety

| Signal | DAQ pin | Notes |
|---|---|---|
| Photodiode | `AI0` (RSE) | Use a photodiode amplifier, or a photodiode with a load resistor, so you get a voltage. Ground goes to AI GND. |
| Fan | not connected | Runs from its own power supply at a fixed speed. |
| Laser | not connected | Runs from its own power supply. |

No analog outputs are used in this notebook. **If the AO0 → AI0 loopback wire from the
exercises is still connected, remove it.** Otherwise AI0 measures the AO output instead of
your photodiode.

⚠️ **Laser safety:** never look into the beam, and watch out for reflections off the fan blades.
Use a low-power module (class 2 or below) and keep the beam path below eye level.
⚠️ Keep fingers away from the spinning fan. It keeps spinning when your code stops,
so switch it off at its own supply before reaching near it.

---
## Why a fan? The neuroscience connection

This setup is a small, safe model of two instruments that neuroscience labs use every day.

### A scanning (multiphoton) microscope

A two-photon microscope has **no camera**. It focuses a pulsed laser to a single point, moves
that point across the tissue with scan mirrors, and records the fluorescence from each
position with a single detector, a photomultiplier tube (PMT). The image only exists
because we know **where the beam was at each moment**: time is converted into position.

Our fan setup does the same thing the other way round. The beam stays still and the object
moves through it. We record the photodiode over time and use the known speed to turn time into distance:

$$\text{size} = \text{speed} \times \text{time}$$

That is exactly how a microscope's pixel size is set: µm per pixel = scan speed × time per pixel.
It's also why scan speed has to be known precisely. A **resonant scanner** (typically ~8 kHz)
moves sinusoidally, so it travels fastest in the middle of each line and slows at the edges,
and imaging software has to correct for that or the image is distorted at the edges. Measuring the fan's
rotation from the blade period T is like measuring a resonant scanner's actual frequency
from its sync output.

### An electrophysiology rig

Look at the photodiode trace and think "electrode": a steady baseline interrupted by brief,
stereotyped events. Finding the blade pulses is the same problem as **spike detection**:

* set a threshold relative to the baseline and the noise
* find the threshold crossings
* cut out a short snippet around each crossing and overlay them, to see the event's shape
* measure each event's **width** and the **interval** between events

The photodiode itself is standard kit in **optogenetics**. A photodiode picking off some of
the stimulation light is how labs check that the light came on when it was commanded, and
for as long. Here the laser runs from its own supply, so we only use the measuring side of that.

### How the parts map

| This experiment | Two-photon microscope | Ephys / optogenetics rig |
|---|---|---|
| Laser beam | Excitation laser | Optogenetic laser/LED |
| Fan blades moving through the beam | Scan mirrors moving the beam over the tissue | — |
| Photodiode on AI0 | PMT detecting fluorescence | Electrode + amplifier, or a light-monitoring photodiode |
| Dark level (blade fully blocking the beam) | PMT offset, subtracted from images | Baseline |
| Blade pulse | Beam passing over a labelled cell | A spike |
| 1000 snippets overlaid and averaged | Averaging repeated frames | Spike waveforms overlaid; a spike-triggered average |
| Pulse width τ → blade width | Time per pixel → pixel size | Spike width |
| Blade period T | Line period of the scanner | Inter-spike interval → firing rate |
| Hardware sample clock: sample *i* is at *i* / `RATE` | Pixel clock: sample *i* is pixel *i* | The amplifier's fixed sampling rate |

The 🧠 notes through the notebook point out where each step shows up in a real experiment.

---
## 0. Setup

Leave `SIMULATE = True` to run the whole notebook without hardware. It generates a
realistic fake photodiode signal, so you can check the analysis first.
Set it to `False` once everything is wired up.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

SIMULATE = True   # <-- set False to use the real DAQ

### Configuration

Everything you may need to change is in this one cell.

In [ ]:
# --- DAQ channel ---
DEV = "Dev1"           # device name, as shown in NI MAX
PD_CHAN = f"{DEV}/ai0"

# --- Acquisition ---
RATE = 20_000          # samples/s. Check your device's maximum (e.g. USB-6001 is 20 kS/s)
THRESHOLD_V = 1.0      # a blade arriving makes the signal fall through this (V)
PRE, POST = 500, 500   # samples kept before and after each crossing
N_SNIPPETS = 1000      # blade passes per recording
MIN_GAP_S = 0.5e-3     # ignore crossings this soon after the last one (noise on the same edge)
MAX_SECONDS = 30.0     # give up if there aren't enough blade passes by then

# --- The fan (measure these!) ---
N_BLADES = 7
BEAM_RADIUS_MM = 30.0  # distance from the fan's axis to the laser beam

# --- Analysis ---
EDGE_FIT_MS = 0.5      # fit each Gaussian to the slope within this time of its edge

### Fake data for `SIMULATE = True`

This models a fan spinning at about 20 revolutions per second, with blades about 15 mm wide
(each slightly different), a Gaussian laser beam 1 mm across, a slight wobble in speed and a
little noise. Like a real fan on its own supply, its speed varies by a few percent from one
recording to the next. It has nothing to do with DAQmx. It's only there so the analysis can
be tested against a known answer.

In [ ]:
SIM_TRUE_BLADE_MM = 15.0
SIM_BEAM_DIAMETER_MM = 1.0  # 1/e² diameter of the Gaussian beam
SIM_FAN_HZ = 20.0           # rotation frequency (revolutions per second)
SIM_SPEED_JITTER = 0.03     # 3 % speed variation between recordings
SIM_DARK_V, SIM_LIGHT_V = 0.05, 2.0
rng = np.random.default_rng(0)


def fake_reader(rate, chunk):
    """Returns a function that reads the next `chunk` samples of a fake fan."""
    blade_mm = SIM_TRUE_BLADE_MM + rng.normal(0, 0.15, N_BLADES)     # the blades differ slightly
    speed = 2 * np.pi * BEAM_RADIUS_MM * SIM_FAN_HZ * (1 + rng.normal(0, SIM_SPEED_JITTER))  # mm/s
    pitch = 2 * np.pi * BEAM_RADIUS_MM / N_BLADES                    # mm from one blade to the next
    start = rng.uniform(0, 100 * pitch)
    sigma = SIM_BEAM_DIAMETER_MM / 4
    n_done = 0

    def read_chunk():
        nonlocal n_done
        t = (n_done + np.arange(chunk)) / rate
        n_done += chunk
        x = start + speed * t + 0.01 * speed / (2 * np.pi) * np.sin(2 * np.pi * t)  # mm travelled
        k = np.floor((x + (pitch - SIM_TRUE_BLADE_MM) / 2) / pitch)  # nearest blade
        s = x - k * pitch                                            # beam position from its leading edge
        w = blade_mm[k.astype(int) % N_BLADES]
        blocked = (erf(s / (np.sqrt(2) * sigma)) - erf((s - w) / (np.sqrt(2) * sigma))) / 2
        light = SIM_LIGHT_V - (SIM_LIGHT_V - SIM_DARK_V) * blocked
        return light + rng.normal(0, 0.02, chunk)

    return read_chunk

---
## 1. Checking the signal: live view (continuous AI)

Switch the laser and the fan on at their own power supplies. While adjusting the laser and
photodiode, it helps to watch the signal live. This is the continuous loop from tutorial
section 4b.

* If you can switch the fan off, first align for the **highest** signal with the beam unblocked.
* With the fan spinning, you should see a regular train of dips. The top of the trace should
  stay high, and the dips should go close to zero when a blade fully blocks the beam.

In [ ]:
from IPython.display import clear_output, display


def live_view(seconds=10.0, chunk=5_000):
    """Plot the photodiode signal live for a while (continuous AI)."""
    if SIMULATE:
        print("[sim] live view needs hardware; skipping")
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * chunk
        )
        task.start()
        t_end = time.time() + seconds
        while time.time() < t_end:
            data = np.array(task.read(number_of_samples_per_channel=chunk))
            ax.clear()
            ax.plot(np.arange(chunk) / RATE * 1000, data)
            ax.set_ylim(-0.5, 10)
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Photodiode (V)")
            ax.set_title(f"Mean {data.mean():.3f} V")
            clear_output(wait=True)
            display(fig)
    plt.close(fig)


live_view(seconds=10)

---
## 2. The measurement: a snippet around every blade pass

The DAQ streams continuously and we read it in chunks, as in tutorial section 4a. After each
chunk we look for places where the signal **falls through** `THRESHOLD_V`, i.e. a blade
arriving at the beam. For each one we keep `PRE` samples before the crossing and `POST`
samples after it, until we have `N_SNIPPETS` of them.

Set `THRESHOLD_V` roughly halfway between the dark and light levels you saw in the live
view. Its exact value doesn't matter for the result: section 4 finds the edges by fitting,
not from the threshold.

The DAQ's own clock times every sample, so sample *i* was taken at exactly *i* / `RATE`
seconds. That's what lets us turn sample counts into times, and line snippets up to within
a sample.

> **🧠 In the lab:** this is exactly how many ephys acquisition systems store spikes. They
> stream continuously, detect threshold crossings, and save a short snippet (typically 1–2 ms)
> around each one. Overlaying the snippets shows the waveform's shape, and averaging them
> (a *spike-triggered average*) pulls a clean shape out of noisy single events. The noise
> falls by √N: by about 30× for 1000 snippets.

In [ ]:
def find_crossings(signal, threshold):
    """Sample indices where `signal` falls through `threshold` (a blade arriving)."""
    return np.flatnonzero((signal[:-1] >= threshold) & (signal[1:] < threshold)) + 1


def collect(read_chunk, rate, pre, post):
    """Read chunks until there are N_SNIPPETS complete snippets.

    Returns (snippets, crossings, data): one row of `snippets` per blade pass, and the
    sample index of each crossing in `data`.
    """
    data = np.empty(0)
    crossings = np.empty(0, dtype=int)
    t_end = time.time() + MAX_SECONDS
    while time.time() < t_end:
        data = np.concatenate([data, read_chunk()])
        crossings = find_crossings(data, THRESHOLD_V)
        # Like a refractory period: keep only the first crossing on a noisy edge.
        crossings = crossings[np.diff(crossings, prepend=-10**9) > MIN_GAP_S * rate]
        crossings = crossings[(crossings >= pre) & (crossings + post <= data.size)]
        if len(crossings) >= N_SNIPPETS:
            crossings = crossings[:N_SNIPPETS]
            snippets = np.stack([data[c - pre:c + post] for c in crossings])
            return snippets, crossings, data
    dark, light = np.percentile(data, [5, 95])
    raise RuntimeError(f"Only {len(crossings)} blade passes in {MAX_SECONDS:.0f} s. Set THRESHOLD_V "
                       f"between the dark ({dark:.2f} V) and light ({light:.2f} V) levels.")


def record(rate=RATE, pre=PRE, post=POST):
    """Record N_SNIPPETS blade passes. Returns (snippets, crossings, data)."""
    chunk = max(rate // 10, 100)   # about 10 reads per second
    if SIMULATE:
        return collect(fake_reader(rate, chunk), rate, pre, post)

    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=rate, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * chunk
        )
        task.start()
        read_chunk = lambda: np.array(task.read(number_of_samples_per_channel=chunk))
        return collect(read_chunk, rate, pre, post)

In [ ]:
snippets, crossings, data = record()

dark, light = np.percentile(data, [5, 95])
print(f"{len(snippets)} blade passes;  dark {dark:.2f} V, light {light:.2f} V")
if light - dark < 0.1:
    print("⚠️  Very little contrast between light and dark: is the fan spinning? Check the alignment with live_view()")

t_ms = np.arange(-PRE, POST) / RATE * 1000
plt.figure(figsize=(10, 4))
plt.plot(t_ms, snippets.T, color="tab:blue", lw=0.5, alpha=0.05)
plt.plot(t_ms, snippets.mean(axis=0), color="k", lw=1.5, label="mean")
plt.axhline(THRESHOLD_V, color="gray", ls=":", label="threshold")
plt.xlabel("Time from threshold crossing (ms)")
plt.ylabel("Photodiode (V)")
plt.title(f"{len(snippets)} blade passes, overlaid")
plt.legend(loc="lower right")
plt.show()

---
## 3. Reading the overlay

* At **t = 0** every snippet falls through the threshold together, so that edge is sharp.
* **Further from t = 0** the edges blur. The fan's speed wobbles slightly and the blades
  aren't all identical, so the later an edge is, the more its timing varies from one blade
  pass to the next. Only the first dip (the blade that triggered the snippet) is used below.
* Moving `THRESHOLD_V` slides every trace sideways so that its crossing is still at t = 0,
  but it doesn't change their shape.

> **🧠 In the lab:** spike waveforms behave the same way. Overlaid on their threshold
> crossing, they're tightly aligned near the crossing and spread out further away, which is
> why spike-sorting and averaging windows are kept short.

---
## 4. Gaussian edges → blade width

1. Average the snippets, then take the slope of the mean (`np.diff` × `RATE`, in V/s).
2. The arriving edge is at t = 0, where the slope is negative: fit a Gaussian to −slope there.
3. The leaving edge is the biggest rise within one blade period: fit a Gaussian there too.
4. **Blade period** T: the mean time between successive crossings.
5. **Rotation frequency** f = 1 / (N·T), **blade speed** v = 2πrf.
6. **Blade width** w = v × (t_leave − t_arrive), **beam diameter** d = 4vσ (from the arriving edge,
   which isn't blurred).

`curve_fit` adjusts the Gaussian's height, centre and width until it matches the slope as closely
as possible (least squares), starting from a rough guess.

In [ ]:
def gaussian(t, amplitude, centre, sigma):
    return amplitude * np.exp(-(t - centre) ** 2 / (2 * sigma ** 2))


def fit_edge(t, slope, around):
    """Fit a Gaussian to `slope` within EDGE_FIT_MS of time `around`. Returns (amplitude, centre, sigma)."""
    dt = t[1] - t[0]
    near = np.abs(t - around) <= max(EDGE_FIT_MS / 1000, 4 * dt)
    guess = [slope[near].max(), around, 2 * dt]
    params, _ = curve_fit(gaussian, t[near], slope[near], p0=guess)
    return params


def blade_period(crossings, rate):
    """Mean time (s) between successive blade arrivals."""
    return np.mean(np.diff(crossings)) / rate


def analyse(snippets, rate, pre, period, n_blades=N_BLADES, radius_mm=BEAM_RADIUS_MM):
    """Fit both edges of the mean snippet. Returns a dict of results (times in s, sizes in mm)."""
    t = (np.arange(snippets.shape[1]) - pre) / rate   # 0 = threshold crossing
    t_mid = t[:-1] + 0.5 / rate                       # times between samples, for np.diff
    mean = snippets.mean(axis=0)
    slope = np.diff(mean) * rate

    a_arrive, t_arrive, s_arrive = fit_edge(t_mid, -slope, around=0.0)
    after = (t_mid > EDGE_FIT_MS / 1000) & (t_mid < period)
    if not after.any() or slope[after].max() < 0.5 * (-slope).max():
        raise RuntimeError("No leaving edge in the snippets. Increase POST.")
    a_leave, t_leave, s_leave = fit_edge(t_mid, slope, around=t_mid[after][np.argmax(slope[after])])

    rot_freq = 1 / (n_blades * period)
    speed = 2 * np.pi * radius_mm * rot_freq
    return dict(t=t, t_mid=t_mid, mean=mean, slope=slope, rot_freq=rot_freq,
                arrive=(-a_arrive, t_arrive, abs(s_arrive)), leave=(a_leave, t_leave, abs(s_leave)),
                width_mm=speed * (t_leave - t_arrive), beam_mm=4 * speed * abs(s_arrive))


period = blade_period(crossings, RATE)
res = analyse(snippets, RATE, PRE, period)

print(f"Rotation:      {res['rot_freq']:.2f} Hz  ({res['rot_freq'] * 60:.0f} rpm)")
print(f"Edges:         arrives {res['arrive'][1] * 1e3:.3f} ms, leaves {res['leave'][1] * 1e3:.3f} ms "
      f"(σ {res['arrive'][2] * 1e3:.3f} and {res['leave'][2] * 1e3:.3f} ms)")
print(f"Blade width:   {res['width_mm']:.2f} mm")
print(f"Beam diameter: {res['beam_mm']:.2f} mm")
if SIMULATE:
    print(f"[sim] true blade width about {SIM_TRUE_BLADE_MM} mm, beam {SIM_BEAM_DIAMETER_MM} mm")

In [ ]:
t_arrive, t_leave = res["arrive"][1], res["leave"][1]
show = (res["t"] > -1e-3) & (res["t"] < t_leave + 1e-3)
show_mid = show[:-1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(res["t"][show] * 1e3, snippets[:200, show].T, color="tab:blue", lw=0.5, alpha=0.05)
ax1.plot(res["t"][show] * 1e3, res["mean"][show], color="k", lw=1.5, label="mean")
ax1.axvspan(t_arrive * 1e3, t_leave * 1e3, color="orange", alpha=0.2, label=f"blade: {res['width_mm']:.2f} mm")
ax1.set_ylabel("Photodiode (V)")
ax1.legend(loc="lower right")
ax1.set_title("Blade passes and their mean")

ax2.plot(res["t_mid"][show_mid] * 1e3, res["slope"][show_mid] / 1e3, color="k", lw=1, label="slope of the mean")
for a, centre, sigma in [res["arrive"], res["leave"]]:
    ax2.plot(res["t_mid"][show_mid] * 1e3, gaussian(res["t_mid"][show_mid], a, centre, sigma) / 1e3,
             color="tab:red", lw=1.5, ls="--")
ax2.plot([], [], color="tab:red", ls="--", label="fitted Gaussians")
ax2.set_xlabel("Time from threshold crossing (ms)")
ax2.set_ylabel("Slope (V/ms)")
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

> **🧠 In the lab:** `speed × time = distance` is the whole calculation behind a scanning
> microscope's pixel size. For example, a resonant scanner at 8 kHz that sweeps 500 µm in roughly
> half of each 125 µs period, sampled into 512 pixels, gives about 1 µm per pixel and about 0.1 µs per
> pixel. In ephys, the same per-event numbers become **spike width** and **firing rate**
> (1 / inter-spike interval). Spike width is often used to tell narrow-spiking, putative
> fast-spiking interneurons apart from broad-spiking, putative pyramidal cells.

The fitted σ is the same measurement as a **knife-edge** scan of a microscope's laser focus:
move a sharp edge across the beam, and the slope of the transmitted light is the beam's profile.

### Each blade separately

Blade passes arrive in a fixed order, so counting blade periods from the first crossing tells
us which blade made each snippet. Fitting each blade's snippets separately shows whether the
blades differ. Which group is "blade 1" is arbitrary, because we don't know which blade passed first.

In [ ]:
blade_index = np.round((crossings - crossings[0]) / (period * RATE)).astype(int) % N_BLADES
per_blade = [analyse(snippets[blade_index == b], RATE, PRE, period)["width_mm"] for b in range(N_BLADES)]

plt.figure(figsize=(6, 3))
plt.plot(range(1, N_BLADES + 1), per_blade, "o")
plt.axhline(res["width_mm"], color="k", ls=":", label="all blades")
plt.xlabel("Blade (arbitrary numbering)")
plt.ylabel("Width (mm)")
plt.title("Width of each blade")
plt.legend()
plt.show()

print("Widths:", np.round(per_blade, 2), "mm")

If one blade is clearly offset from the others, check that `N_BLADES` is correct.
If it's wrong, the groups mix different blades together and the rotation frequency is off by a factor.

Fitting each blade separately also sharpens its leaving edge: within one blade, the blur at
that edge comes only from the fan's speed wobble, not from differences between blades.

> **🧠 In the lab:** sorting events by "which blade made them" is a very simple version of
> **spike sorting**, i.e. working out which neuron each spike on an electrode came from. Here it's
> easy because the blades arrive in a fixed order. With real neurons you group spikes by waveform
> shape instead, and getting the number of groups wrong causes the same kind of mixing.

---
## 5. Checking the measurement

We can't change the fan's speed, but we can still test the measurement in two ways.

### 5a. Is it repeatable?

Record several times in a row. A fan on its own supply doesn't hold a perfectly constant
speed: it drifts a little with warm-up, supply voltage and air flow, so the rpm will differ
slightly between recordings. Because the width depends only on τ/T, it **shouldn't** follow
those changes. If the width moves with the rpm, the edges are being measured wrongly.

In [ ]:
N_REPEATS = 5
repeats = []

for i in range(N_REPEATS):
    snip_i, cross_i, _ = record()
    r = analyse(snip_i, RATE, PRE, blade_period(cross_i, RATE))
    repeats.append((r["rot_freq"] * 60, r["width_mm"], r["beam_mm"]))
    print(f"Recording {i + 1}: {r['rot_freq'] * 60:6.0f} rpm, width {r['width_mm']:6.2f} mm, beam {r['beam_mm']:.2f} mm")

rpm, width, beam = np.array(repeats).T
print(f"\nBlade width over {N_REPEATS} recordings: {width.mean():.2f} ± {width.std():.2f} mm")

### 5b. Is the sample rate high enough?

Each edge only lasts a few samples, and each snippet is lined up on the nearest sample to its
crossing. The Gaussian fit and the averaging over 1000 snippets place the edges to a fraction
of a sample, but only while the edge spans a few samples. Record at several rates, keeping the
snippet the same length in time: the width should level off as the rate goes up, while the
beam diameter is overestimated at low rates, where the edge is narrower than a sample. Keep
every rate at or below your device's maximum.

> **🧠 In the lab:** this is the same choice as an ephys sampling rate. A spike is about 1 ms
> long, so you need ≥20 kHz to see its shape and measure its width, even though a few kHz
> would be enough to just count spikes. In a microscope, the time per pixel sets how many PMT
> samples go into each pixel. A detector or amplifier that is too slow (low bandwidth) has a
> similar effect: it rounds off the blade edges, and the fitted beam looks bigger than it is.

In [ ]:
SAMPLE_RATES = [2_000, 5_000, 10_000, 20_000]   # don't go above your device's maximum
by_rate = []

for r in SAMPLE_RATES:
    snip_r, cross_r, _ = record(rate=r, pre=PRE * r // RATE, post=POST * r // RATE)
    try:
        res_r = analyse(snip_r, r, PRE * r // RATE, blade_period(cross_r, r))
        by_rate.append((res_r["width_mm"], res_r["beam_mm"]))
        print(f"{r:6d} samples/s: width {res_r['width_mm']:6.2f} mm, beam {res_r['beam_mm']:.2f} mm")
    except RuntimeError as err:   # curve_fit raises RuntimeError when it can't converge
        by_rate.append((np.nan, np.nan))
        print(f"{r:6d} samples/s: fit failed ({err})")

rate_width, rate_beam = np.array(by_rate).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(rpm, width, "o")
ax1.axhline(width.mean(), color="k", ls=":")
ax1.set_xlabel("Speed (rpm)")
ax1.set_ylabel("Blade width (mm)")
ax1.set_title("5a: repeated recordings (should be flat)")

ax2.plot(SAMPLE_RATES, rate_width, "o-", label="blade width")
ax2.set_xscale("log")
ax2.set_xlabel("Sample rate (samples/s)")
ax2.set_ylabel("Blade width (mm)")
ax2b = ax2.twinx()
ax2b.plot(SAMPLE_RATES, rate_beam, "s--", color="tab:orange", label="beam diameter")
ax2b.set_ylabel("Beam diameter (mm)", color="tab:orange")
ax2.set_title("5b: results vs sample rate")
plt.tight_layout()
plt.show()

---
## 6. Finishing up

The DAQ doesn't output anything in this notebook, so there is nothing to reset in software.
Switch the laser and the fan off **at their own power supplies**, and wait for the fan to
stop before touching the setup.

## Things to try

* **Measure with a ruler** and compare. Fan blades are angled (pitched), so what the beam measures
  is the blade's width along the direction of rotation at radius r, not along its surface.
* **Move the beam to a different radius.** Many blades get wider towards the tip.
* **One fit per blade pass:** fit the two edges of every snippet separately, plot a histogram of
  the 1000 widths, and fit a Gaussian to the histogram. Its σ is the precision of a single blade
  pass. Compare it with how precisely the mean snippet gives the width.
* **Fit the signal instead of its slope:** fit a cumulative Gaussian (`scipy.special.erf`) to each
  edge of the mean snippet directly. Do you get the same centres and σ?
* **How steady is the fan?** `np.diff(crossings) / RATE` is the time between every pair of blades.
  Plot it against time. This is how you would check a resonant scanner's frequency for drift.
* **Check the beam diameter** by moving a razor blade across the beam on a translation stage
  while watching `live_view()`, and compare it with the fitted value.
* **Make an image:** record for one full rotation and reshape the signal into rows, one per
  rotation, over several rotations. You've built a (very slow) scanning imager. Compare it with
  how a microscope reshapes the PMT signal into a frame.